### Transaction location profiling

In [0]:
%sql
SELECT
    CASE
        WHEN is_online = true
             AND merchant_state IS NULL
             AND merchant_zip IS NULL
            THEN 'A - ONLINE: state and zip missing'

        WHEN is_online = true
             AND (merchant_state IS NOT NULL OR merchant_zip IS NOT NULL)
            THEN 'A - ONLINE: UNEXPECTED LOCATION DATA'

        WHEN is_online = false
             AND merchant_state IS NOT NULL
             AND merchant_state RLIKE '^[A-Za-z]{2}$'
             AND merchant_zip IS NOT NULL
            THEN 'US - state + zip populated'

        WHEN is_online = false
             AND merchant_state IS NOT NULL
             AND NOT (merchant_state RLIKE '^[A-Za-z]{2}$')
             AND merchant_zip IS NULL
            THEN 'B - INTERNATIONAL: country + zip missing'

        WHEN is_online = false
             AND merchant_state IS NULL
             AND merchant_zip IS NULL
            THEN 'UNEXPECTED: offline with no location'

        ELSE 'OTHER / POTENTIAL DATA QUALITY ISSUE'
    END AS location_pattern,
    COUNT(*) AS transaction_count
FROM credit_card_project.silver.cleaned_transactions
GROUP BY
    CASE
        WHEN is_online = true
             AND merchant_state IS NULL
             AND merchant_zip IS NULL
            THEN 'A - ONLINE: state and zip missing'

        WHEN is_online = true
             AND (merchant_state IS NOT NULL OR merchant_zip IS NOT NULL)
            THEN 'A - ONLINE: UNEXPECTED LOCATION DATA'

        WHEN is_online = false
             AND merchant_state IS NOT NULL
             AND merchant_state RLIKE '^[A-Za-z]{2}$'
             AND merchant_zip IS NOT NULL
            THEN 'US - state + zip populated'

        WHEN is_online = false
             AND merchant_state IS NOT NULL
             AND NOT (merchant_state RLIKE '^[A-Za-z]{2}$')
             AND merchant_zip IS NULL
            THEN 'B - INTERNATIONAL: country + zip missing'

        WHEN is_online = false
             AND merchant_state IS NULL
             AND merchant_zip IS NULL
            THEN 'UNEXPECTED: offline with no location'

        ELSE 'OTHER / POTENTIAL DATA QUALITY ISSUE'
    END
ORDER BY transaction_count DESC;


### Merchant identity and location
Does each merchant have exactly one MCC and one location, or can those attributes vary?

In [0]:
%sql

SELECT
    merchant_id,
    COUNT(*) AS transaction_count,
    COUNT(DISTINCT merchant_category_code) AS distinct_mccs,
    COUNT(DISTINCT merchant_city) AS distinct_cities,
    COUNT(DISTINCT merchant_state) AS distinct_states,
    COUNT(DISTINCT merchant_zip) AS distinct_zips
FROM credit_card_project.silver.cleaned_transactions
GROUP BY merchant_id
HAVING
    COUNT(DISTINCT merchant_category_code) > 1
    OR COUNT(DISTINCT merchant_city) > 1
    OR COUNT(DISTINCT merchant_state) > 1
    OR COUNT(DISTINCT merchant_zip) > 1
ORDER BY transaction_count DESC;


Result: For these high-volume merchants, distinct_mccs = 1 consistently, while cities, states, and ZIPs can vary dramatically.

Next: Is MCC stable for a merchant across the entire dataset?

In [0]:
%sql
SELECT
    COUNT(*) AS total_merchants,
    SUM(CASE WHEN distinct_mccs = 1 THEN 1 ELSE 0 END) AS merchants_single_mcc,
    SUM(CASE WHEN distinct_mccs > 1 THEN 1 ELSE 0 END) AS merchants_multiple_mccs,
    MAX(distinct_mccs) AS max_mccs_per_merchant
FROM (
    SELECT
        merchant_id,
        COUNT(DISTINCT merchant_category_code) AS distinct_mccs
    FROM credit_card_project.silver.cleaned_transactions
    GROUP BY merchant_id
) m;


Result: So merchant_id → MCC is almost, but not completely, one-to-one.

Are above 654 exceptions (merchants_multiple_mccs) genuinely associated with two MCCs or is there's some temporal/data pattern behind it?

In [0]:
%sql
SELECT
    merchant_id,
    merchant_category_code,
    COUNT(*) AS transaction_count,
    MIN(transaction_timestamp) AS first_transaction,
    MAX(transaction_timestamp) AS last_transaction
FROM credit_card_project.silver.cleaned_transactions
WHERE merchant_id IN (
    SELECT merchant_id
    FROM (
        SELECT
            merchant_id,
            COUNT(DISTINCT merchant_category_code) AS distinct_mccs
        FROM credit_card_project.silver.cleaned_transactions
        GROUP BY merchant_id
    )
    WHERE distinct_mccs > 1
)
GROUP BY
    merchant_id,
    merchant_category_code
ORDER BY
    merchant_id,
    transaction_count DESC;


Finding: merchant → MCC is not strictly one-to-one, so merchant_category_code should remain a transaction-level analytical attribute and we should not put MCC into a merchant dimension as though it were a fixed merchant attribute.

=> merchant_id alone and its metrics should aggregate transactions regardless of MCC

Does every transaction have a merchant ID?

In [0]:
%sql
SELECT
    COUNT(*) AS total_transactions,
    COUNT(merchant_id) AS transactions_with_merchant_id,
    COUNT(*) - COUNT(merchant_id) AS transactions_without_merchant_id,
    COUNT(DISTINCT merchant_id) AS distinct_merchants
FROM credit_card_project.silver.cleaned_transactions;


Merchant profiling conclusion:
- 24,386,900 / 24,386,900 transactions have a merchant_id.
- There are 100,343 distinct merchants.
- Merchant IDs can be negative or positive; sign has no business meaning.
- A merchant can transact across many cities, states/countries, and ZIP codes.
- merchant_id is therefore the appropriate merchant identifier.
- merchant_category_code is not strictly a fixed merchant attribute because 654 merchants have multiple MCCs.
- MCC should therefore remain a transaction-level/category attribute, rather than being treated as a permanent property of the merchant.![](path)

### Next profiling target: users
profile of the user → card → transaction relationships.

Does every user have at least one card, and does every card have transactions?

In [0]:
%sql
SELECT
    COUNT(*) AS total_users,
    COUNT(CASE WHEN card_count > 0 THEN 1 END) AS users_with_cards,
    COUNT(CASE WHEN card_count = 0 THEN 1 END) AS users_without_cards,
    MIN(card_count) AS min_cards_per_user,
    MAX(card_count) AS max_cards_per_user,
    AVG(card_count) AS avg_cards_per_user
FROM (
    SELECT
        u.user_id,
        COUNT(c.card_id) AS card_count
    FROM credit_card_project.bronze.users u
    LEFT JOIN credit_card_project.bronze.cards c
        ON u.user_id = c.user_id
    GROUP BY u.user_id
) x;


User → Card profiling conclusion

We can now establish:

- 2,000 / 2,000 users have at least one card
- No users without cards
- Minimum cards per user: 1
- Maximum: 9
- Average: 3.073 cards per user

In [0]:
%sql
SELECT
    COUNT(*) AS total_cards,
    COUNT(CASE WHEN transaction_count > 0 THEN 1 END) AS cards_with_transactions,
    COUNT(CASE WHEN transaction_count = 0 THEN 1 END) AS cards_without_transactions,
    MIN(transaction_count) AS min_transactions_per_card,
    MAX(transaction_count) AS max_transactions_per_card,
    AVG(transaction_count) AS avg_transactions_per_card
FROM (
    SELECT
        c.user_id,
        c.card_id,
        COUNT(t.card_id) AS transaction_count
    FROM credit_card_project.bronze.cards c
    LEFT JOIN credit_card_project.silver.cleaned_transactions t
        ON c.user_id = t.user_id
        AND c.card_id = t.card_id
    GROUP BY
        c.user_id,
        c.card_id
) x;


Card profiling conclusion

We have:

- 6,146 total cards
- 6,139 cards with transactions
- 7 cards with zero transactions
- Minimum transactions per card: 0
- Maximum: 70,008
- Average: 3,967.93

So there are exactly 7 cards that exist in the source but have never appeared in the transaction data.
=> That means gold_card_behavior should be card-driven, not transaction-driven.

### Profile card-attribute stability

In [0]:
%sql
SELECT
    user_id,
    card_id,
    COUNT(*) AS card_rows,
    COUNT(DISTINCT card_brand) AS distinct_brands,
    COUNT(DISTINCT card_type) AS distinct_types,
    COUNT(DISTINCT expires) AS distinct_expiry_dates,
    COUNT(DISTINCT acct_open_date) AS distinct_open_dates,
    COUNT(DISTINCT credit_limit) AS distinct_credit_limits
FROM credit_card_project.bronze.cards
GROUP BY
    user_id,
    card_id
HAVING
    COUNT(*) > 1
    OR COUNT(DISTINCT card_brand) > 1
    OR COUNT(DISTINCT card_type) > 1
    OR COUNT(DISTINCT expires) > 1
    OR COUNT(DISTINCT acct_open_date) > 1
    OR COUNT(DISTINCT credit_limit) > 1
ORDER BY card_rows DESC;


=> (user_id, card_id) is a stable logical card identifier

Next: Are the attributes such as card_brand, card_type, credit_limit, and the two dates are sufficiently populated and sensible?

In [0]:
%sql
SELECT
    COUNT(*) AS total_card_rows,
    COUNT(DISTINCT CONCAT(user_id, '_', card_id)) AS distinct_logical_cards,

    COUNT(card_brand) AS cards_with_brand,
    COUNT(card_type) AS cards_with_type,
    COUNT(credit_limit) AS cards_with_credit_limit,
    COUNT(acct_open_date) AS cards_with_open_date,
    COUNT(expires) AS cards_with_expiry,

    COUNT(DISTINCT card_brand) AS distinct_brands,
    COUNT(DISTINCT card_type) AS distinct_types,
    COUNT(DISTINCT credit_limit) AS distinct_credit_limits,

    MIN(credit_limit) AS min_credit_limit,
    MAX(credit_limit) AS max_credit_limit
FROM credit_card_project.bronze.cards;


conclusions:
- (user_id, card_id) is the natural logical card key and is unique in the current card population
- card attributes are completely populated => for these no null handling logic needed 
- for an eventual card-level gold dataset: a gold table gold_card_behavior can use grain = one row per (user_id, card_id)
- manageable cardinality: brands = 4 and types = 3
- credit_limit $0 may have legitimate meaning, needs more profiling

In [0]:
%sql
SELECT
    credit_limit,
    COUNT(*) AS card_count
FROM credit_card_project.bronze.cards
GROUP BY credit_limit
ORDER BY credit_limit;


=> field is a numeric source attribute, rather than a predefined set of standard credit-limit tiers

In [0]:
%sql
SELECT
    card_brand,
    card_type,
    COUNT(*) AS card_count
FROM credit_card_project.bronze.cards
GROUP BY
    card_brand,
    card_type
ORDER BY
    card_brand,
    card_type;


- so both are descriptive attributes
- Up to this point we haven't found a reason to introduce surrogate card_key values or SCD handling

In [0]:
%sql
SELECT
    CASE
        WHEN expires RLIKE '^[0-9]{2}/[0-9]{4}$'
            THEN 'MM/YYYY'
        WHEN expires RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$'
            THEN 'MM/DD/YYYY'
        ELSE 'OTHER'
    END AS expires_format,
    COUNT(*) AS card_count
FROM credit_card_project.bronze.cards
GROUP BY
    CASE
        WHEN expires RLIKE '^[0-9]{2}/[0-9]{4}$'
            THEN 'MM/YYYY'
        WHEN expires RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$'
            THEN 'MM/DD/YYYY'
        ELSE 'OTHER'
    END
ORDER BY card_count DESC;


In [0]:
%sql
SELECT
    CASE
        WHEN acct_open_date RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$'
            THEN 'MM/DD/YYYY'
        WHEN acct_open_date RLIKE '^[0-9]{2}/[0-9]{4}$'
            THEN 'MM/YYYY'
        ELSE 'OTHER'
    END AS acct_open_date_format,
    COUNT(*) AS card_count
FROM credit_card_project.bronze.cards
GROUP BY
    CASE
        WHEN acct_open_date RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$'
            THEN 'MM/DD/YYYY'
        WHEN acct_open_date RLIKE '^[0-9]{2}/[0-9]{4}$'
            THEN 'MM/YYYY'
        ELSE 'OTHER'
    END
ORDER BY card_count DESC;


That means I would not create card_age_at_transaction or days_until_card_expiration as currently defined. Those names imply day-level precision that the source doesn't provide.

### Next profiling target: card activity by card type
Does transaction behavior differs meaningfully across the card types we've identified?

In [0]:
%sql
SELECT
    c.card_brand,
    c.card_type,
    COUNT(*) AS card_count,
    SUM(
        CASE WHEN t.card_id IS NOT NULL THEN 1 ELSE 0 END
    ) AS cards_with_transactions,
    SUM(
        CASE WHEN t.card_id IS NULL THEN 1 ELSE 0 END
    ) AS cards_without_transactions,
    COUNT(t.card_id) AS transaction_count,
    COALESCE(SUM(t.amount), 0) AS total_amount,
    COALESCE(AVG(t.amount), 0) AS avg_transaction_amount
FROM credit_card_project.bronze.cards c
LEFT JOIN credit_card_project.silver.cleaned_transactions t
    ON c.user_id = t.user_id
   AND c.card_id = t.card_id
GROUP BY
    c.card_brand,
    c.card_type
ORDER BY
    c.card_brand,
    c.card_type;


-- This reinforces the eventual Gold pattern:
- bronze.cards LEFT JOIN transaction aggregates

-- There is a clear difference in average transaction amount:
- The distinction between ordinary Debit and Debit (Prepaid) is particularly strong

-- Brand itself appears less important for transaction amount


Fraud behavior by card type:

In [0]:
%sql
SELECT
    c.card_brand,
    c.card_type,
    COUNT(t.card_id) AS transaction_count,
    SUM(
        CASE
            WHEN t.is_fraud = true THEN 1
            ELSE 0
        END
    ) AS fraud_count,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN t.is_fraud = true THEN 1
                ELSE 0
            END
        ) / NULLIF(COUNT(t.card_id), 0),
        4
    ) AS fraud_rate_pct
FROM credit_card_project.bronze.cards c
LEFT JOIN credit_card_project.silver.cleaned_transactions t
    ON c.user_id = t.user_id
   AND c.card_id = t.card_id
GROUP BY
    c.card_brand,
    c.card_type
ORDER BY
    fraud_rate_pct DESC;


Architectural consequence: proposed card-level metrics:
- total_transactions
- total_spend
- avg_transaction_amount
- fraud_count
- fraud_rate
- unique_merchants
- unique_mccs
- online_transaction_pct
- weekend_transaction_pct
- night_transaction_pct



### Next profiling question
Were unique_merchants and unique_mccs be genuinely informative card-behavior metrics?

In [0]:
%sql
SELECT
    c.card_brand,
    c.card_type,
    COUNT(*) AS card_count,
    ROUND(AVG(COALESCE(a.unique_merchants, 0)), 2) AS avg_unique_merchants,
    MAX(COALESCE(a.unique_merchants, 0)) AS max_unique_merchants,
    ROUND(AVG(COALESCE(a.unique_mccs, 0)), 2) AS avg_unique_mccs,
    MAX(COALESCE(a.unique_mccs, 0)) AS max_unique_mccs
FROM credit_card_project.bronze.cards c
LEFT JOIN (
    SELECT
        user_id,
        card_id,
        COUNT(DISTINCT merchant_id) AS unique_merchants,
        COUNT(DISTINCT merchant_category_code) AS unique_mccs
    FROM credit_card_project.silver.cleaned_transactions
    GROUP BY
        user_id,
        card_id
) a
    ON c.user_id = a.user_id
   AND c.card_id = a.card_id
GROUP BY
    c.card_brand,
    c.card_type
ORDER BY
    c.card_brand,
    c.card_type;


Prepaid cards appear to have substantially narrower merchant/category diversity than credit and ordinary debit cards.

In [0]:
%sql
SELECT
    c.card_brand,
    c.card_type,
    COUNT(*) AS card_count,
    ROUND(
        AVG(
            COALESCE(a.online_transaction_pct, 0)
        ),
        2
    ) AS avg_online_transaction_pct,
    MIN(
        COALESCE(a.online_transaction_pct, 0)
    ) AS min_online_transaction_pct,
    MAX(
        COALESCE(a.online_transaction_pct, 0)
    ) AS max_online_transaction_pct
FROM credit_card_project.bronze.cards c
LEFT JOIN (
    SELECT
        user_id,
        card_id,
        100.0 * AVG(
            CASE
                WHEN is_online = true THEN 1.0
                ELSE 0.0
            END
        ) AS online_transaction_pct
    FROM credit_card_project.silver.cleaned_transactions
    GROUP BY
        user_id,
        card_id
) a
    ON c.user_id = a.user_id
   AND c.card_id = a.card_id
GROUP BY
    c.card_brand,
    c.card_type
ORDER BY
    c.card_brand,
    c.card_type;


Card-level range is wide => online_transaction_pct is a reasonable behavioral metric for gold_card_behavior

Current gold_card_behavior candidate set

Card attributes:
- user_id
- card_id
- card_brand
- card_type
- credit_limit
- acct_open_date
- expires

Activity:
- total_transactions
- total_spend
- avg_transaction_amount

Diversity:
- unique_merchants
- unique_mccs

Fraud:
- fraud_count
- fraud_rate

Behavior:
- online_transaction_pct
- weekend_transaction_pct
- night_transaction_pct



### Final card-behavior profiling step
Any unexpected result for weekend and night behavior together?

In [0]:
%sql
DESCRIBE credit_card_project.silver.cleaned_transactions;


These are valid candidate behavioral metrics:
- online_transaction_pct
- weekend_transaction_pct
- night_transaction_pct


### Card profiling is done
gold_card_behavior


Grain: (user_id, card_id)

Attributes:
-   card_brand
-   card_type
-   credit_limit
-   acct_open_date
-   expires

Activity:
-   total_transactions
-   total_spend
-   avg_transaction_amount

Diversity:
-   unique_merchants
-   unique_mccs

Fraud:
-   fraud_count
-   fraud_rate

Behavior:
-   online_transaction_pct
-   weekend_transaction_pct
-   night_transaction_pct


### Gold_user_behavior profiling
Do users actually differ meaningfully in transaction volume and spending behavior?

In [0]:
%sql
SELECT
    COUNT(*) AS user_count,

    MIN(transaction_count) AS min_transactions,
    ROUND(AVG(transaction_count), 2) AS avg_transactions,
    MAX(transaction_count) AS max_transactions,

    MIN(total_amount) AS min_total_amount,
    ROUND(AVG(total_amount), 2) AS avg_total_amount,
    MAX(total_amount) AS max_total_amount,

    MIN(avg_transaction_amount) AS min_avg_transaction_amount,
    ROUND(AVG(avg_transaction_amount), 2) AS avg_transaction_amount,
    MAX(avg_transaction_amount) AS max_avg_transaction_amount

FROM (
    SELECT
        user_id,
        COUNT(*) AS transaction_count,
        SUM(amount) AS total_amount,
        AVG(amount) AS avg_transaction_amount
    FROM credit_card_project.silver.cleaned_transactions
    GROUP BY user_id
) u;


So a simple user-level aggregate would clearly add analytical meaning beyond Silver.

Strong candidates for gold_user_behavior:
- total_transactions
- total_spend
- avg_transaction_amount

The distribution is particularly important because it shows that users aren't homogeneous. A user with 15 transactions and ~$136 total amount represents a very different behavioral profile from one with 82K transactions and ~$4.48M.


### Next user profiling question: card and merchant diversity at the user level

In [0]:
%sql
SELECT
    MIN(unique_cards) AS min_unique_cards,
    ROUND(AVG(unique_cards), 2) AS avg_unique_cards,
    MAX(unique_cards) AS max_unique_cards,

    MIN(unique_merchants) AS min_unique_merchants,
    ROUND(AVG(unique_merchants), 2) AS avg_unique_merchants,
    MAX(unique_merchants) AS max_unique_merchants,

    MIN(unique_mccs) AS min_unique_mccs,
    ROUND(AVG(unique_mccs), 2) AS avg_unique_mccs,
    MAX(unique_mccs) AS max_unique_mccs

FROM (
    SELECT
        user_id,
        COUNT(DISTINCT card_id) AS unique_cards,
        COUNT(DISTINCT merchant_id) AS unique_merchants,
        COUNT(DISTINCT merchant_category_code) AS unique_mccs
    FROM credit_card_project.silver.cleaned_transactions
    GROUP BY user_id
) u;


- This confirms the diversity metrics are meaningful at the user level too
- This matches our earlier Bronze-card profiling and confirms that unique_cards is a useful user-level behavioral measure
- A user's merchant breadth clearly tells us something different from simply counting transactions
- A user can have many transactions without necessarily having the same degree of merchant/category diversity

So the current gold_user_behavior candidates are looking solid:
- total_transactions
- total_spend
- avg_transaction_amount
- unique_cards
- unique_merchants
- unique_mccs


### Next: user fraud behavior

In [0]:
%sql
SELECT
    COUNT(*) AS user_count,

    MIN(fraud_count) AS min_fraud_count,
    ROUND(AVG(fraud_count), 2) AS avg_fraud_count,
    MAX(fraud_count) AS max_fraud_count,

    MIN(fraud_rate_pct) AS min_fraud_rate_pct,
    ROUND(AVG(fraud_rate_pct), 4) AS avg_fraud_rate_pct,
    MAX(fraud_rate_pct) AS max_fraud_rate_pct

FROM (
    SELECT
        user_id,
        SUM(
            CASE
                WHEN is_fraud = true THEN 1
                ELSE 0
            END
        ) AS fraud_count,

        100.0 * SUM(
            CASE
                WHEN is_fraud = true THEN 1
                ELSE 0
            END
        ) / COUNT(*) AS fraud_rate_pct

    FROM credit_card_project.silver.cleaned_transactions
    GROUP BY user_id
) u;


Well-supported candidates for gold_user_behavior:
- total_transactions
- total_spend
- avg_transaction_amount
- unique_cards
- unique_merchants
- unique_mccs
- fraud_count
- fraud_rate


### Final check of temporal behavior
For night behavior, the definition we used was: 00:00–05:59

In [0]:
%sql
SELECT
    MIN(online_transaction_pct) AS min_online_pct,
    ROUND(AVG(online_transaction_pct), 2) AS avg_online_pct,
    MAX(online_transaction_pct) AS max_online_pct,

    MIN(weekend_transaction_pct) AS min_weekend_pct,
    ROUND(AVG(weekend_transaction_pct), 2) AS avg_weekend_pct,
    MAX(weekend_transaction_pct) AS max_weekend_pct,

    MIN(night_transaction_pct) AS min_night_pct,
    ROUND(AVG(night_transaction_pct), 2) AS avg_night_pct,
    MAX(night_transaction_pct) AS max_night_pct

FROM (
    SELECT
        user_id,

        100.0 * AVG(
            CASE
                WHEN is_online = true THEN 1.0
                ELSE 0.0
            END
        ) AS online_transaction_pct,

        100.0 * AVG(
            CASE
                WHEN DAYOFWEEK(transaction_timestamp) IN (1, 7)
                    THEN 1.0
                ELSE 0.0
            END
        ) AS weekend_transaction_pct,

        100.0 * AVG(
            CASE
                WHEN HOUR(transaction_timestamp) < 6
                    THEN 1.0
                ELSE 0.0
            END
        ) AS night_transaction_pct

    FROM credit_card_project.silver.cleaned_transactions
    GROUP BY user_id
) u;


There is meaningful variation across users for all three temporal/online behaviors, so all three can provide analytical value

Proposed gold_user_behavior:

one row per user_id

Core attributes/metrics:
- user_id
- total_transactions
- total_spend
- avg_transaction_amount
- fraud_count
- fraud_rate
- unique_cards
- unique_merchants
- unique_mccs
- online_transaction_pct
- weekend_transaction_pct
- night_transaction_pct


### Understand the transaction timeline

Is transaction_timestamp sufficiently populated and what time range does the transaction population cover?

In [0]:
%sql
SELECT
    COUNT(*) AS total_transactions,
    COUNT(transaction_timestamp) AS transactions_with_timestamp,
    COUNT(*) - COUNT(transaction_timestamp) AS transactions_without_timestamp,
    MIN(transaction_timestamp) AS earliest_transaction,
    MAX(transaction_timestamp) AS latest_transaction,
    COUNT(DISTINCT DATE(transaction_timestamp)) AS distinct_transaction_dates
FROM credit_card_project.silver.cleaned_transactions;


- So every transaction can participate in temporal feature calculations
- That is more than enough temporal coverage for the candidate: 1-hour/24-hour/30-day lookback

Are transactions chronologically ordered within a user/card, or are there multiple transactions at exactly the same timestamp? If so which one is "before" the other?

In [0]:
%sql
SELECT
    COUNT(*) AS duplicate_user_card_timestamps,
    SUM(transaction_count) AS transactions_in_duplicate_groups,
    MAX(transaction_count) AS max_transactions_at_same_timestamp
FROM (
    SELECT
        user_id,
        card_id,
        transaction_timestamp,
        COUNT(*) AS transaction_count
    FROM credit_card_project.silver.cleaned_transactions
    GROUP BY
        user_id,
        card_id,
        transaction_timestamp
    HAVING COUNT(*) > 1
) d;


=> Transaction_timestamp alone does not provide a unique chronological ordering of transactions for a card

Are transactions occurring before the current transaction on the same timestamp but with different cards.

In [0]:
%sql
SELECT
    COUNT(*) AS user_timestamp_groups_with_multiple_cards,
    SUM(transaction_count) AS transactions_in_these_groups,
    MAX(transaction_count) AS max_transactions_at_same_timestamp
FROM (
    SELECT
        user_id,
        transaction_timestamp,
        COUNT(*) AS transaction_count,
        COUNT(DISTINCT card_id) AS distinct_cards
    FROM credit_card_project.silver.cleaned_transactions
    GROUP BY
        user_id,
        transaction_timestamp
    HAVING COUNT(DISTINCT card_id) > 1
) d;


Now we have the information needed to define our temporal boundary correctly. Strictly earlier timestamps are the safe boundary when we don't have a finer-grained event ordering.

### Next profiling step: transaction ordering by user/card

Does every user/card have a meaningful sequence of transactions before later transactions?

In [0]:
%sql
WITH tx AS (
    SELECT
        user_id,
        card_id,
        transaction_timestamp,

        LAG(transaction_timestamp) OVER (
            PARTITION BY user_id
            ORDER BY transaction_timestamp
        ) AS previous_user_timestamp,

        LAG(transaction_timestamp) OVER (
            PARTITION BY user_id, card_id
            ORDER BY transaction_timestamp
        ) AS previous_card_timestamp

    FROM credit_card_project.silver.cleaned_transactions
)

SELECT
    'user' AS history_level,
    COUNT(*) AS transaction_count,
    SUM(
        CASE
            WHEN previous_user_timestamp IS NOT NULL
                THEN 1
            ELSE 0
        END
    ) AS transactions_with_prior_timestamp
FROM tx

UNION ALL

SELECT
    'card' AS history_level,
    COUNT(*) AS transaction_count,
    SUM(
        CASE
            WHEN previous_card_timestamp IS NOT NULL
                THEN 1
            ELSE 0
        END
    ) AS transactions_with_prior_timestamp
FROM tx;


=> vast majority of transactions have prior activity at both levels

### Next step: profile the actual lookback windows

Before deciding that all proposed windows (1h, 24h, 30 days) belong in gold_fraud_features, let's measure how much historical activity users actually have inside those windows.

In [0]:
%sql
SELECT
    MIN(transactions_last_24h) AS min_transactions_last_24h,
    ROUND(AVG(transactions_last_24h), 2) AS avg_transactions_last_24h,
    MAX(transactions_last_24h) AS max_transactions_last_24h,

    SUM(
        CASE
            WHEN transactions_last_24h > 0 THEN 1
            ELSE 0
        END
    ) AS transactions_with_prior_24h_activity

FROM (
    SELECT
        user_id,
        transaction_timestamp,

        COUNT(*) OVER (
            PARTITION BY user_id
            ORDER BY UNIX_TIMESTAMP(transaction_timestamp)
            RANGE BETWEEN 86400 PRECEDING AND 1 PRECEDING
        ) AS transactions_last_24h

    FROM credit_card_project.silver.cleaned_transactions
) t;


That means user_transaction_count_24h is a meaningful fraud/behavior feature.

Next: 1-hour velocity

In [0]:
%sql
SELECT
    MIN(transactions_last_1h) AS min_transactions_last_1h,
    ROUND(AVG(transactions_last_1h), 4) AS avg_transactions_last_1h,
    MAX(transactions_last_1h) AS max_transactions_last_1h,

    SUM(
        CASE
            WHEN transactions_last_1h > 0 THEN 1
            ELSE 0
        END
    ) AS transactions_with_prior_1h_activity

FROM (
    SELECT
        user_id,
        transaction_timestamp,

        COUNT(*) OVER (
            PARTITION BY user_id
            ORDER BY UNIX_TIMESTAMP(transaction_timestamp)
            RANGE BETWEEN 3600 PRECEDING AND 1 PRECEDING
        ) AS transactions_last_1h

    FROM credit_card_project.silver.cleaned_transactions
) t;


So about 28.3% of transactions have at least one prior transaction by the same user within the preceding hour.

One final profiling check: 30-day activity

In [0]:
%sql
SELECT
    MIN(transactions_last_30d) AS min_transactions_last_30d,
    ROUND(AVG(transactions_last_30d), 2) AS avg_transactions_last_30d,
    MAX(transactions_last_30d) AS max_transactions_last_30d,

    SUM(
        CASE
            WHEN transactions_last_30d > 0 THEN 1
            ELSE 0
        END
    ) AS transactions_with_prior_30d_activity

FROM (
    SELECT
        user_id,
        transaction_timestamp,

        COUNT(*) OVER (
            PARTITION BY user_id
            ORDER BY UNIX_TIMESTAMP(transaction_timestamp)
            RANGE BETWEEN 2592000 PRECEDING AND 1 PRECEDING
        ) AS transactions_last_30d

    FROM credit_card_project.silver.cleaned_transactions
) t;


Only 2,612 transactions have no prior 30-day activity

### One last profiling check

Can we calculate a meaningful user's historical average transaction amount and standard deviation using only transactions strictly before the current transaction?

In [0]:
%sql
WITH tx AS (
    SELECT
        user_id,
        transaction_timestamp,
        amount,

        AVG(amount) OVER (
            PARTITION BY user_id
            ORDER BY UNIX_TIMESTAMP(transaction_timestamp)
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS user_avg_amount_before_txn,

        STDDEV_SAMP(amount) OVER (
            PARTITION BY user_id
            ORDER BY UNIX_TIMESTAMP(transaction_timestamp)
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS user_std_amount_before_txn

    FROM credit_card_project.silver.cleaned_transactions
)

SELECT
    MIN(user_avg_amount_before_txn) AS min_historical_avg,
    ROUND(AVG(user_avg_amount_before_txn), 2) AS avg_historical_avg,
    MAX(user_avg_amount_before_txn) AS max_historical_avg,

    MIN(user_std_amount_before_txn) AS min_historical_std,
    ROUND(AVG(user_std_amount_before_txn), 2) AS avg_historical_std,
    MAX(user_std_amount_before_txn) AS max_historical_std,

    SUM(
        CASE
            WHEN user_avg_amount_before_txn IS NOT NULL
                THEN 1
            ELSE 0
        END
    ) AS transactions_with_historical_avg

FROM tx;


24,384,900 / 24,386,900 transactions have a prior historical average. The 2,000 without one correspond naturally to the first transaction of each user. So these are analytically viable: user_avg_amount_before_txn, user_std_amount_before_txn

### Profiling is  complete

User behavior

Supported:
- total_transactions
- total_spend
- avg_transaction_amount
- fraud_count
- fraud_rate
- unique_cards
- unique_merchants
- unique_mccs
- online_transaction_pct
- weekend_transaction_pct
- night_transaction_pct

Card behavior

Supported:
- (user_id, card_id) grain
- stable card attributes
- transaction activity
- spend
- fraud
- merchant/MCC diversity
- online behavior


Merchant behavior

Supported:
- merchant_id grain
- transaction volume
- spend
- fraud
- user/card diversity
- MCC diversity
- location diversity
- online behavior


Fraud features

We established:
- timestamps are complete
- transaction period is 1991–2020
- same-timestamp transactions exist
- cross-card same-timestamp transactions exist
- therefore timestamp alone cannot establish an ordering among tied transactions
- historical features should use strictly earlier timestamps
- 1h, 24h and 30d windows all provide meaningful variation
- historical user amount baselines are available for essentially all non-first transactions